In [7]:
import pandas as pd
import json

# 1. Load the raw data from the data folder
file_path = '../data/raw_credit_applications.json'
with open(file_path, 'r') as file:
    data = json.load(file)

# 2. Flatten the nested 'applicant_info' so it's easy to read in a table
df = pd.json_normalize(data)

# 3. Define our Pseudonymization/Anonymization functions
def mask_ssn(ssn):
    """Masks all but the last 4 digits of an SSN to protect PII."""
    if pd.isna(ssn) or not isinstance(ssn, str):
        return ssn
    return "XXX-XX-" + ssn[-4:] if len(ssn) >= 4 else "XXX-XX-XXXX"

def mask_ip(ip):
    """Masks the last octet of an IPv4 address for data minimization."""
    if pd.isna(ip) or not isinstance(ip, str):
        return ip
    parts = ip.split('.')
    if len(parts) == 4:
        return f"{parts[0]}.{parts[1]}.{parts[2]}.XXX"
    return ip

# --- NEW: Advanced Data Minimization (Generalization) ---
def categorize_age(dob):
    """Generalizes DOB into age bands to comply with GDPR Art. 5."""
    try:
        birth_year = pd.to_datetime(dob).year
        current_year = 2026 # Per project setting
        age = current_year - birth_year
        if age < 25: return "18-24"
        if age < 35: return "25-34"
        if age < 45: return "35-44"
        if age < 55: return "45-54"
        return "55+"
    except:
        return "Unknown"

def minimize_zip(zip_code):
    """Reduces granularity of ZIP code to prevent re-identification."""
    if pd.isna(zip_code): return zip_code
    return str(zip_code)[:3] + "XX"

# --- BONUS: Sensitive Behavioral Data Minimization ---
def sanitize_spending(spending_list):
    """Summarizes spending behavior to remove sensitive category labels."""
    if not isinstance(spending_list, list): return 0
    # We count 'High Risk' categories instead of naming them (Data Minimization)
    sensitive_categories = ['Gambling', 'Alcohol', 'Adult Entertainment']
    risk_count = sum(1 for item in spending_list if item.get('category') in sensitive_categories)
    return risk_count

if 'spending_behavior' in df.columns:
    df['high_risk_spending_count'] = df['spending_behavior'].apply(sanitize_spending)

# 4. Apply the masks and generalizations
if 'applicant_info.ssn' in df.columns:
    df['applicant_info.ssn_masked'] = df['applicant_info.ssn'].apply(mask_ssn)
    
if 'applicant_info.ip_address' in df.columns:
    df['applicant_info.ip_address_masked'] = df['applicant_info.ip_address'].apply(mask_ip)

if 'applicant_info.date_of_birth' in df.columns:
    df['age_band'] = df['applicant_info.date_of_birth'].apply(categorize_age)

if 'applicant_info.zip_code' in df.columns:
    df['zip_masked'] = df['applicant_info.zip_code'].apply(minimize_zip)

# 5. Display the "Before and After" to prove it to your professor
columns_to_show = [
    'applicant_info.full_name', 
    'applicant_info.ssn', 'applicant_info.ssn_masked', 
    'applicant_info.ip_address', 'applicant_info.ip_address_masked',
    'applicant_info.date_of_birth', 'age_band',
    'applicant_info.zip_code', 'zip_masked'
]

existing_cols = [col for col in columns_to_show if col in df.columns]

print("Privacy & Governance Demo: Pseudonymization and Data Generalization")
df[existing_cols].head()

Privacy & Governance Demo: Pseudonymization and Data Generalization


C:\Users\prate\AppData\Local\Temp\ipykernel_17732\1916549252.py:32: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  birth_year = pd.to_datetime(dob).year


,applicant_info.full_name,applicant_info.ssn,applicant_info.ssn_masked,applicant_info.ip_address,applicant_info.ip_address_masked,applicant_info.date_of_birth,age_band,applicant_info.zip_code,zip_masked
0,Jerry Smith,596-64-4340,XXX-XX-4340,192.168.48.155,192.168.48.XXX,2001-03-09,25-34,10036,100XX
1,Brandon Walker,425-69-4784,XXX-XX-4784,10.1.102.112,10.1.102.XXX,1992-03-31,25-34,10032,100XX
2,Scott Moore,370-78-5178,XXX-XX-5178,10.240.193.250,10.240.193.XXX,1989-10-24,35-44,10075,100XX
3,Thomas Lee,194-35-1833,XXX-XX-1833,192.168.175.67,192.168.175.XXX,1983-04-25,35-44,10077,100XX
4,Brian Rodriguez,480-41-2475,XXX-XX-2475,172.29.125.105,172.29.125.XXX,1999-05-21,25-34,10080,100XX
